# AraBERT v02 reproduction — full Colab training

This notebook runs the configured 10-epoch baseline using the plain PyTorch training loop. It trains on the frozen train CSV and validates on the frozen validation CSV. It does not load or evaluate the test CSV.

## Before running

1. In Colab, select **Runtime → Change runtime type → T4 GPU** (or another CUDA GPU). The trainer will use CUDA when available and CPU otherwise.
2. Put the complete project folder in Google Drive, including `src/`, `configs/`, and `data/splits/arsarcasm_v2/{train,validation}.csv`. The next cell expects it at `/content/drive/MyDrive/mizan-arabert-sentiment`; edit that path if yours differs.
3. Keep the project folder in Drive so checkpoints and `history.csv` persist if the runtime stops.

The baseline uses mixed precision off, as specified in the config. To resume after a runtime interruption, rerun setup and change `resume_run` to `True` in the training cell. The last completed epoch is restored from the last checkpoint.

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [4]:
from pathlib import Path
import os
import sys
from google.colab import drive

drive.mount('/content/drive')
project_root = Path('/content/drive/MyDrive/mizan-arabert-sentiment')
os.chdir(project_root)
sys.path.insert(0, str(project_root))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [5]:
%pip install -q -r requirements-bonus.txt

## Run the baseline

The default config sets AraBERT v02, 3 labels, Adam (`lr=1e-5`, `eps=1e-8`, weight decay `0`), batch size 40, max length 256, and 10 epochs. Training writes per-epoch train and validation metrics to `checkpoints/bonus/repro_arabertv02_base/history.csv`, with best and last checkpoints in the same folder.

In [6]:
from src.bonus.train import run_training

config_path = 'configs/bonus/repro_arabertv02_base.yaml'
resume_run = False
history = run_training(config_path, resume=resume_run)
history

config.json:   0%|          | 0.00/384 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/381 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/825k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.64M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  543MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: aubmindlab/bert-base-arabertv02
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
/content

epoch 1/10: train_loss=0.7295 validation_loss=0.6197 validation_macro_f_pn=0.6922
epoch 2/10: train_loss=0.5573 validation_loss=0.6329 validation_macro_f_pn=0.6848
epoch 3/10: train_loss=0.4677 validation_loss=0.5986 validation_macro_f_pn=0.7062
epoch 4/10: train_loss=0.3828 validation_loss=0.6571 validation_macro_f_pn=0.6997
epoch 5/10: train_loss=0.3024 validation_loss=0.7341 validation_macro_f_pn=0.6802
epoch 6/10: train_loss=0.2327 validation_loss=0.7810 validation_macro_f_pn=0.6899
epoch 7/10: train_loss=0.1802 validation_loss=0.8711 validation_macro_f_pn=0.6943
epoch 8/10: train_loss=0.1443 validation_loss=0.9260 validation_macro_f_pn=0.6966
epoch 9/10: train_loss=0.1201 validation_loss=0.9931 validation_macro_f_pn=0.6979
epoch 10/10: train_loss=0.0956 validation_loss=1.1001 validation_macro_f_pn=0.7108


,epoch,train_accuracy,train_weighted_precision,train_weighted_recall,train_weighted_f1,train_macro_f_pn,train_loss,train_NEG_precision,train_NEG_recall,train_NEG_f1,...,validation_NEG_precision,validation_NEG_recall,validation_NEG_f1,validation_NEU_precision,validation_NEU_recall,validation_NEU_f1,validation_POS_precision,validation_POS_recall,validation_POS_f1,validation_confusion_matrix
0,1,0.691667,0.684492,0.691667,0.680315,0.582767,0.729499,0.699978,0.762924,0.730097,...,0.709369,0.803030,0.753299,0.801541,0.723478,0.760512,0.638498,0.623853,0.631090,"[[371, 64, 27], [109, 416, 50], [43, 39, 136]]"
1,2,0.773045,0.772049,0.773045,0.772270,0.731727,0.557340,0.783249,0.818466,0.800470,...,0.814536,0.703463,0.754936,0.717873,0.845217,0.776358,0.681564,0.559633,0.614610,"[[325, 115, 22], [54, 486, 35], [20, 76, 122]]"
2,3,0.814398,0.813795,0.814398,0.813908,0.784762,0.467722,0.825956,0.856937,0.841161,...,0.745418,0.792208,0.768101,0.765542,0.749565,0.757469,0.671642,0.619266,0.644391,"[[366, 76, 20], [98, 431, 46], [27, 56, 135]]"
3,4,0.856106,0.855795,0.856106,0.855839,0.838620,0.382839,0.869220,0.894927,0.881886,...,0.788506,0.742424,0.764771,0.747588,0.808696,0.776942,0.666667,0.605505,0.634615,"[[343, 93, 26], [70, 465, 40], [22, 64, 132]]"
4,5,0.889046,0.888828,0.889046,0.888910,0.875576,0.302421,0.906933,0.918490,0.912675,...,0.750524,0.774892,0.762513,0.731908,0.773913,0.752325,0.682353,0.532110,0.597938,"[[358, 88, 16], [92, 445, 38], [27, 75, 116]]"
5,6,0.916320,0.916138,0.916320,0.916194,0.905640,0.232707,0.930288,0.940130,0.935183,...,0.747433,0.787879,0.767123,0.758037,0.779130,0.768439,0.683616,0.555046,0.612658,"[[364, 79, 19], [90, 448, 37], [33, 64, 121]]"
6,7,0.939609,0.939506,0.939609,0.939540,0.930281,0.180232,0.949488,0.958163,0.953806,...,0.768707,0.733766,0.750831,0.742718,0.798261,0.769489,0.673469,0.605505,0.637681,"[[339, 98, 25], [77, 459, 39], [25, 61, 132]]"
7,8,0.952448,0.952389,0.952448,0.952406,0.945414,0.144305,0.959476,0.967781,0.963610,...,0.779279,0.748918,0.763797,0.768142,0.754783,0.761404,0.593496,0.669725,0.629310,"[[346, 81, 35], [76, 434, 65], [22, 50, 146]]"
8,9,0.959710,0.959659,0.959710,0.959674,0.952430,0.120064,0.967634,0.970426,0.969028,...,0.773913,0.770563,0.772234,0.756897,0.763478,0.760173,0.627907,0.619266,0.623557,"[[356, 84, 22], [78, 439, 58], [26, 57, 135]]"
9,10,0.970867,0.970848,0.970867,0.970843,0.966620,0.095558,0.973659,0.977639,0.975645,...,0.731898,0.809524,0.768756,0.791271,0.725217,0.756806,0.654378,0.651376,0.652874,"[[374, 65, 23], [106, 417, 52], [31, 45, 142]]"


## Review saved results

The final training cell returns the complete history table. The same rows are saved to `history.csv` after each completed epoch, so they remain available after a disconnected runtime. The notebook stops after training and validation; any held-out test evaluation remains a separate explicit action outside this notebook.